# Analyse de la Déforestation par Traitement d'Images

**Client** : Agronome Forestier
**Objectif** : Détecter et quantifier l'évolution de la forêt entre deux dates (t0 et t1).

Ce notebook présente un pipeline modulaire, rigoureux et justifié, conçu pour transformer des images satellites en données exploitables pour la gestion forestière.

## 0. Configuration de l'Environnement
Importation des modules métiers développés spécifiquement pour ce projet.

In [ ]:
import sys
import os
# On s'assure d'être à la racine du projet
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.path.abspath('src'))

import src.data_loader as dl
import src.visualization as vis
import src.preprocessing as pre
import src.features as ft
import src.clustering as cl
import src.postprocessing as post
import src.quantification as quant

import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

# Pipeline d'Analyse Automatisé par Zone

Cette fonction encapsule les 11 étapes demandées pour permettre un traitement systématique de toutes les parcelles forestières.

In [ ]:

def run_full_pipeline(p0, p1, name, k=3):
    print(f"\n{'='*60}")
    print(f"ANALYSE DE LA ZONE : {name}")
    print(f"{'='*60}")
    
    # 0 & 1. Acquisition et Chargement
    img_a, img_b = dl.load_pair((p0, p1))
    
    # 2. Prétraitement
    img_a_p = pre.preprocess(img_a)
    img_b_p = pre.preprocess(img_b)
    
    # 3 & 4. Extraction de Features
    f0 = ft.get_feature_vector(img_a_p)
    f1 = ft.get_feature_vector(img_b_p)
    
    # 6 & 7. Segmentation et Identification Végétation
    l0, c0 = cl.run_kmeans(f0, n_clusters=k)
    v0 = cl.identify_vegetation_cluster(f0, l0, c0)
    m0 = cl.get_vegetation_mask(l0, v0, img_a.shape[:2])
    
    l1, c1 = cl.run_kmeans(f1, n_clusters=k)
    v1 = cl.identify_vegetation_cluster(f1, l1, c1)
    m1 = cl.get_vegetation_mask(l1, v1, img_b.shape[:2])
    
    # 8. Post-traitement
    m0_c = post.apply_morphology(m0)
    m1_c = post.apply_morphology(m1)
    
    # 9. Quantification
    s = quant.compute_deforestation_stats(m0_c, m1_c)
    s['image'] = name  # Ajout du nom pour le tableau recap
    
    # 10. Visualisation
    vis.plot_side_by_side(img_a, img_b, title_a="t0 (Avant)", title_b="t1 (Après)", main_title=f"Visualisation Initiale - {name}")
    vis.plot_side_by_side(m0_c, m1_c, title_a="Masque t0", title_b="Masque t1", main_title=f"Masques de Végétation Nettoyés - {name}")
    
    change_map = quant.get_change_map(m0_c, m1_c)
    vis.plot_change_map(change_map)
    
    # 11. Interprétation (Verdict)
    loss = s['loss_percentage']
    verdict = "OUI" if loss > 2 else "NON (ou négligeable)"
    ampleur = "Critique" if loss > 10 else ("Significative" if loss > 5 else "Modérée")
    
    print(f"--- VERDICT POUR LA ZONE {name} ---")
    print(f"Déforestation détectée : {verdict}")
    print(f"Perte de surface : {loss:.2f}%")
    if loss > 2:
        print(f"Ampleur de la perte : {ampleur}")
        print(f"Note : Les zones rouges sur la carte indiquent les coupes rases ou dégradations.")
    else:
        print(f"Note : La forêt est stable ou en régénération sur cette zone.")
    print(f"Limites de la méthode : Sensibilité aux ombres et aux nuages portés.")
    
    return s


## Exécution du Rapport Complet

Nous lançons l'analyse sur toutes les paires d'images disponibles.

In [ ]:

all_stats = []
for p0, p1 in pairs:
    name = os.path.basename(p0)
    stat = run_full_pipeline(p0, p1, name)
    all_stats.append(stat)

df = pd.DataFrame(all_stats)
df = df[['image', 'surface_t0', 'surface_t1', 'loss_percentage']]
df.sort_values(by='loss_percentage', ascending=False, inplace=True)


# Tableau de Bord Global de la Forêt
Synthèse des performances et des pertes par zone.

In [ ]:
df.style.background_gradient(cmap='Reds', subset=['loss_percentage'])